# Lesson 4A — Implementing a Controller

*ESP2110 Inverted Pendulum Lab*

**Run in Google Colab:** open the notebook, run the **Setup** cell once, then run
cells top-to-bottom. No local files are required.

## Learning objectives
By the end of this notebook you can:
1. Read the **open-loop eigenvalues** of the linearized cart-pole and explain why the upright pole is unstable.
2. Implement a **PID controller** and reason about the *sign* of feedback needed to balance the pole.
3. Close the loop and compare the controller on the **linear model** vs the **nonlinear plant**, side by side.
4. Identify where the small-angle (linear) approximation breaks down.

### Parameters used in this lesson
| Symbol | Meaning | Value |
| --- | --- | --- |
| `m_c` | Cart mass | 0.5 kg |
| `m_p` | Pole mass | 0.2 kg |
| `L` | Pole length | 0.3 m |
| `g` | Gravity | 9.81 m/s^2 |
| `dt` | Sample time | 0.01 s |

State vector: $x = [\,p,\ \dot p,\ \theta,\ \dot\theta\,]$ with $\theta = 0$ at the **upright** position.

In [ ]:
# --- Setup (safe to re-run) ---
# Installs are only needed on a fresh Colab runtime; locally these are usually present.
try:
    import numpy, scipy, matplotlib  # noqa: F401
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy', 'scipy', 'matplotlib'], check=True)
print('Environment ready.')

---
## Lesson content

### Recap: the model
From Lessons 2-3, the cart-pole has nonlinear dynamics (state $[p,\dot p,\theta,\dot\theta]$, input force $f$):

$$\ddot p = \frac{f + m_p\sin\theta\,(L\dot\theta^2 - g\cos\theta)}{m_c + m_p\sin^2\theta},\qquad\ddot\theta = \frac{-f\cos\theta - m_p L\dot\theta^2\sin\theta\cos\theta + (m_c+m_p)g\sin\theta}{L\,(m_c + m_p\sin^2\theta)}.$$

Linearizing about the upright equilibrium ($\sin\theta\approx\theta,\ \cos\theta\approx1$, small $\dot\theta$) gives
$\dot x = Ax + Bf$ with the $A,B$ defined in code below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Physical parameters
m_c, m_p, L, g, dt = 0.5, 0.2, 0.3, 9.81, 0.01

# Nonlinear plant: returns the state derivative xdot for state s and force f.
def f_nonlin(s, f):
    p, v, th, om = s
    sin, cos = np.sin(th), np.cos(th)
    den = m_c + m_p * sin**2
    vdot  = (f + m_p * sin * (L * om**2 - g * cos)) / den
    omdot = (-f * cos - m_p * L * om**2 * sin * cos + (m_c + m_p) * g * sin) / (L * den)
    return np.array([v, vdot, om, omdot])

# Linearized model about the upright equilibrium (theta = 0)
A = np.array([
    [0, 1, 0,                       0],
    [0, 0, -m_p * g / m_c,          0],
    [0, 0, 0,                       1],
    [0, 0, (m_c + m_p) * g / (L * m_c), 0],
])
B = np.array([0, 1 / m_c, 0, -1 / (L * m_c)])
print('A =\n', A)
print('B =', B)

## Part 1 - Open-loop stability (eigenvalues)

With no controller ($f=0$), the linear dynamics are $\dot x = Ax$. The eigenvalues of $A$ tell
us whether small disturbances grow or decay.

**Task:** compute the eigenvalues of `A` and decide whether the upright pole is stable.

In [ ]:
# TODO: compute the eigenvalues of A and print the maximum real part.
# Hint: np.linalg.eigvals(...)
eigvals = ...  # <-- your code here
print('Open-loop eigenvalues:', eigvals)

One eigenvalue is **positive real** ($\approx +6.77$): any tiny tilt grows exponentially,
so the upright pole is **open-loop unstable**. The other pair sits at the origin (the cart is a
free integrator). This is why we need feedback control.

## Part 2 - A PID controller, and getting the sign right

We drive the pole angle $\theta$ to zero with a PID law on the angle:

$$f = K_p\,\theta + K_i\!\int\theta\,dt + K_d\,\dot\theta.$$

> **Sign warning (read this!).** For the *inverted* pendulum the stabilizing feedback is
> **positive** in $\theta$: when the pole tips forward you must drive the cart *toward* the fall to
> get back underneath it. A conventional 'negative-feedback-on-error' sign ($f=-K_p\theta$) makes
> things **worse** and the closed loop diverges. We verify this with the closed-loop eigenvalues below.

In [ ]:
class PIDController:
    def __init__(self, Kp, Ki, Kd, dt):
        self.Kp, self.Ki, self.Kd, self.dt = Kp, Ki, Kd, dt
        self.integral = 0.0

    def update(self, theta, omega):
        # TODO: accumulate the integral of theta, then return the PID force.
        #   f = Kp*theta + Ki*integral + Kd*omega   (note the stabilizing positive sign)
        ...  # <-- your code here

## Part 3 - Closed-loop simulators (linear vs nonlinear)

Two roll-outs that share the *same* controller: one steps the **linear** model $\dot x = Ax+Bf$,
the other steps the **nonlinear** plant. Comparing them shows how good the linear approximation is.

In [ ]:
def simulate(plant, controller, x0, T=5.0):
    n = int(T / dt)
    x = np.array(x0, dtype=float)
    X = np.zeros((n, 4)); F = np.zeros(n)
    for k in range(n):
        f = controller.update(x[2], x[3])
        X[k], F[k] = x, f
        if plant == 'linear':
            x = x + dt * (A @ x + B * f)
        else:
            x = x + dt * f_nonlin(x, f)
    t = np.arange(n) * dt
    return t, X, F

## Part 4 - Tune the gains and compare

**Task:** choose PID gains that stabilize the pole, then run both plants from a small initial tilt
($\theta_0 = 0.2$ rad $\approx 11^\circ$).

In [ ]:
# TODO: pick stabilizing gains. Try Kp in [10, 40], Kd in [2, 8], Ki = 0 to start.
Kp, Ki, Kd = ..., ..., ...

x0 = [0.0, 0.0, 0.2, 0.0]
tl, Xl, Fl = simulate('linear',    PIDController(Kp, Ki, Kd, dt), x0)
tn, Xn, Fn = simulate('nonlinear', PIDController(Kp, Ki, Kd, dt), x0)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].plot(tl, np.rad2deg(Xl[:, 2]), label='linear')
ax[0].plot(tn, np.rad2deg(Xn[:, 2]), '--', label='nonlinear')
ax[0].set(title='Pole angle', xlabel='time (s)', ylabel='theta (deg)'); ax[0].legend(); ax[0].grid(True)
ax[1].plot(tl, Xl[:, 0], label='linear'); ax[1].plot(tn, Xn[:, 0], '--', label='nonlinear')
ax[1].set(title='Cart position', xlabel='time (s)', ylabel='p (m)'); ax[1].legend(); ax[1].grid(True)
ax[2].plot(tl, Fl, label='linear'); ax[2].plot(tn, Fn, '--', label='nonlinear')
ax[2].set(title='Control force', xlabel='time (s)', ylabel='f (N)'); ax[2].legend(); ax[2].grid(True)
fig.suptitle('Closed loop from theta0 = 0.2 rad'); fig.tight_layout(); plt.show()

**Expected output.** With `Kp=20, Ki=0, Kd=3` the closed-loop eigenvalues are
**0, 0, -6.47, -13.53**. The two **negative real** eigenvalues are the *angle* modes
(well damped, no oscillation), and the **double pole at 0** is the cart: its position is
**not regulated** here, so the overall max real part is 0. The **pole angle** is driven to
~0 within a couple of seconds, and the **linear and nonlinear** curves nearly overlap at this
small angle. The cart, however, **drifts** - that is the motivation for Lesson 4B. If your pole
*diverges*, check the **sign** of the feedback first.

## Part 5 - Where the linear approximation breaks down

Re-run from a larger tilt ($\theta_0 = 0.5$ rad $\approx 29^\circ$) and watch the linear and
nonlinear curves separate.

In [ ]:
# TODO: repeat the comparison from a larger initial angle (theta0 = 0.5 rad)
#       and plot linear vs nonlinear theta on the same axes.
...

The two curves now visibly differ early in the transient: the linear model overshoots/undershoots
because $\sin\theta$ and $\cos\theta$ are no longer $\approx\theta$ and $\approx1$. Both still
recover here, but the linear *prediction* is no longer trustworthy far from upright.

## Part 6 - Watch it balance (animation)

A quick visual sanity check: animate the **nonlinear** cart-pole under your controller. You should
see the pole snap upright and stay there while the cart slides sideways (the cart is unregulated
here, so some drift is expected).

In [ ]:
from matplotlib import animation
from IPython.display import HTML

# Roll out the nonlinear plant, then animate it (works in Colab; no ffmpeg needed).
t, X, F = simulate('nonlinear', PIDController(Kp, Ki, Kd, dt), [0.0, 0.0, 0.2, 0.0])
frames = X[::10]            # ~50 frames for a 5 s run at dt = 0.01

fig, ax = plt.subplots(figsize=(6, 3))
ax.set_ylim(-0.15, 0.55); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('Cart-pole (nonlinear plant)'); ax.set_xlabel('x (m)')
cart = plt.Rectangle((-0.12, 0.0), 0.24, 0.12, fc='steelblue', ec='k'); ax.add_patch(cart)
pole, = ax.plot([], [], lw=3, color='crimson')
bob,  = ax.plot([], [], 'o', color='crimson', ms=9)

def _update(i):
    p, th = frames[i, 0], frames[i, 2]
    cart.set_xy((p - 0.12, 0.0))
    tip_x, tip_y = p + L * np.sin(th), 0.12 + L * np.cos(th)
    pole.set_data([p, tip_x], [0.12, tip_y])
    bob.set_data([tip_x], [tip_y])
    ax.set_xlim(p - 0.8, p + 0.8)
    return cart, pole, bob

anim = animation.FuncAnimation(fig, _update, frames=len(frames), interval=50, blit=False)
plt.close(fig)                 # prevent a duplicate static frame from showing
HTML(anim.to_jshtml())         # use the play button under the figure

---
## Checkpoints
- Open-loop `A` has a **positive** real eigenvalue (~+6.77) -> unstable.
- Closed-loop **angle** modes are two **negative real** eigenvalues (~-6.47 and -13.53);
  the cart contributes a double pole at 0 (unregulated), so the overall max real part is ~0.
- From 0.2 rad the pole settles to ~0 and **linear ~ nonlinear**; the cart drifts.
- From 0.5 rad the linear and nonlinear angle curves **visibly diverge** in the transient.

## Common pitfalls
- **Wrong feedback sign.** `f = -Kp*theta` (textbook negative-error sign) *destabilizes* the
  inverted pole. Use the positive sign and confirm with the closed-loop eigenvalues.
- **Forgetting the cart drifts.** This controller only regulates the angle; a steady cart offset is
  expected here and is fixed in Lesson 4B.
- **Euler step too large.** `dt` much above 0.01 s can make the explicit integrator itself unstable.
- **Reusing controller state.** Build a fresh `PIDController` per run so the integral term resets.